# HITO 3 - Modelado, experimentación y validación
## Proyecto MIMIC-TRIAGE (adaptado a PhysioNet/CinC 2012)

Indice:
1) Preparación de datos y reutilización del dataset procesado
2) Inspección estructural mínima del dataset
3) Distribuciones, cobertura temporal y missingness
4) Correlaciones y análisis por tipo de UCI
5) Baseline experimental
6) Comparación inicial de modelos
7) Validación cruzada estratificada
8) Validación cruzada orientada a ranking clínico
9) Ajuste de hiperparámetros del modelo seleccionado
10) Interpretabilidad del modelo final
11) Conclusiones


## 0) Importacion de librerias


In [2]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
import shap

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

ROOT = Path(".").resolve()
HITO2_ROOT = ROOT.parent / "Hito 2"

DATASET_ROOT = HITO2_ROOT / "predicting-mortality-of-icu-patients-the-physionet-computing-in-cardiology-challenge-2012-1.0.0"
SET_A_DIR = DATASET_ROOT / "set-a"
OUTCOMES_A_PATH = DATASET_ROOT / "Outcomes-a.txt"

CSVS_DIR = HITO2_ROOT / "csvs"
FIGURES_DIR = ROOT / "figures"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CSVS_DIR.mkdir(parents=True, exist_ok=True)


def parse_time_to_hours(time_series: pd.Series) -> pd.Series:
    parts = time_series.astype(str).str.split(":", expand=True)
    return parts[0].astype(int) + (parts[1].astype(int) / 60.0)


def read_set_file(file_path: Path) -> pd.DataFrame:
    df = pd.read_csv(file_path)
    required_cols = {"Time", "Parameter", "Value"}
    missing_cols = required_cols.difference(df.columns)
    if missing_cols:
        raise ValueError(f"{file_path} missing columns: {sorted(missing_cols)}")

    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
    df = df.dropna(subset=["Value"]).copy()
    df["Hours"] = parse_time_to_hours(df["Time"])
    df["RecordID"] = int(file_path.stem)
    return df[["RecordID", "Time", "Hours", "Parameter", "Value"]]


def validate_existing_csvs(csv_dir: Path) -> pd.DataFrame:
    suspicious_cols = {"Length_of_stay", "Survival", "SAPS-I", "SOFA"}
    ignored_files = {
        "processed_features_48h_setA.csv",
        "csv_validation_report.csv",
        "hito2_data_audit_setA.json",
    }

    rows = []
    for csv_file in sorted(csv_dir.glob("*.csv")):
        if csv_file.name in ignored_files:
            continue

        try:
            df = pd.read_csv(csv_file)
            columns = list(df.columns)
            row = {
                "file": csv_file.name,
                "n_rows": int(len(df)),
                "n_cols": int(len(columns)),
                "has_recordid": "RecordID" in columns,
                "has_label_in_hospital_death": "In-hospital_death" in columns,
                "contains_potential_leakage_fields": any(col in suspicious_cols for col in columns),
                "notes": "",
            }

            if row["n_rows"] != 4000:
                row["notes"] = "No parece tabla por estancia de Set A (4000 filas)."
            elif not row["has_recordid"]:
                row["notes"] = "No incluye RecordID."
            elif row["contains_potential_leakage_fields"]:
                row["notes"] = "Incluye variables potencialmente post-ingreso/outcome (revisar leakage)."

            rows.append(row)

        except Exception as exc:
            rows.append(
                {
                    "file": csv_file.name,
                    "n_rows": -1,
                    "n_cols": -1,
                    "has_recordid": False,
                    "has_label_in_hospital_death": False,
                    "contains_potential_leakage_fields": False,
                    "notes": f"No se pudo leer: {exc}",
                }
            )

    report = pd.DataFrame(rows)
    report.to_csv(csv_dir / "csv_validation_report.csv", index=False)
    return report


def build_processed_features(raw_48h: pd.DataFrame, outcomes: pd.DataFrame) -> pd.DataFrame:
    raw_48h = raw_48h.sort_values(["RecordID", "Parameter", "Hours"]).copy()
    grouped = raw_48h.groupby(["RecordID", "Parameter"])["Value"]

    features_long = grouped.agg(
        first="first",
        last="last",
        mean="mean",
        min="min",
        max="max",
        std=lambda x: float(np.std(x.to_numpy(), ddof=0)),
        n_mediciones="count",
    ).reset_index()
    features_long["flag_medido"] = 1

    metrics = [
        "first",
        "last",
        "mean",
        "min",
        "max",
        "std",
        "n_mediciones",
        "flag_medido",
    ]

    wide_parts = []
    for metric in metrics:
        pivot_metric = features_long.pivot(
            index="RecordID",
            columns="Parameter",
            values=metric,
        )
        pivot_metric.columns = [f"{col}_{metric}" for col in pivot_metric.columns]
        wide_parts.append(pivot_metric)

    feature_matrix = pd.concat(wide_parts, axis=1).reset_index()

    n_cols = [c for c in feature_matrix.columns if c.endswith("_n_mediciones")]
    flag_cols = [c for c in feature_matrix.columns if c.endswith("_flag_medido")]
    for col in n_cols + flag_cols:
        feature_matrix[col] = feature_matrix[col].fillna(0).astype(int)

    processed = outcomes[["RecordID", "In-hospital_death"]].merge(
        feature_matrix, on="RecordID", how="left"
    )
    processed = processed.sort_values("RecordID").reset_index(drop=True)
    return processed


def export_figures(raw_48h: pd.DataFrame, processed: pd.DataFrame, outcomes: pd.DataFrame, figures_dir: Path) -> None:
    plt.style.use("ggplot")

    flag_cols = [c for c in processed.columns if c.endswith("_flag_medido")]
    missing_df = (
        pd.DataFrame(
            {
                "Parameter": [c.replace("_flag_medido", "") for c in flag_cols],
                "Presencia_pct": [processed[c].mean() * 100 for c in flag_cols],
            }
        )
        .sort_values("Presencia_pct")
        .reset_index(drop=True)
    )
    missing_df["Missing_pct"] = 100 - missing_df["Presencia_pct"]

    fig, ax = plt.subplots(figsize=(11, 10))
    ax.barh(missing_df["Parameter"], missing_df["Missing_pct"], color="#4c78a8")
    ax.set_title("Missingness por variable (Set A, ventana 0-48h)")
    ax.set_xlabel("% sin medicion en 0-48h")
    ax.set_ylabel("Variable")
    fig.tight_layout()
    fig.savefig(figures_dir / "missingness_setA_0_48h.png", dpi=160)
    plt.close(fig)

    hourly_counts = (
        raw_48h.assign(hour_bin=raw_48h["Hours"].astype(int))
        .groupby("hour_bin")
        .size()
        .reindex(range(49), fill_value=0)
    )
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(hourly_counts.index, hourly_counts.values, color="#f58518", linewidth=2)
    ax.set_title("Cobertura temporal de mediciones (Set A, 0-48h)")
    ax.set_xlabel("Hora desde ingreso")
    ax.set_ylabel("Numero de mediciones")
    fig.tight_layout()
    fig.savefig(figures_dir / "temporal_coverage_setA_0_48h.png", dpi=160)
    plt.close(fig)

    volume_by_param = raw_48h.groupby("Parameter").size().sort_values(ascending=False).head(20)
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.bar(volume_by_param.index, volume_by_param.values, color="#54a24b")
    ax.set_title("Top 20 variables por volumen de mediciones (0-48h)")
    ax.set_xlabel("Variable")
    ax.set_ylabel("Numero de mediciones")
    ax.tick_params(axis="x", rotation=75)
    fig.tight_layout()
    fig.savefig(figures_dir / "measurement_volume_top20_setA_0_48h.png", dpi=160)
    plt.close(fig)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    age_col = "Age_first"
    if age_col in processed.columns:
        valid_age = processed[age_col].where((processed[age_col] >= 0) & (processed[age_col] <= 120))
        axes[0].hist(valid_age.dropna(), bins=20, color="#4c78a8")
    axes[0].set_title("Edad (Age)")
    axes[0].set_xlabel("Anios")

    gender_col = "Gender_first"
    if gender_col in processed.columns:
        gender_counts = processed[gender_col].fillna(-1).value_counts().sort_index()
        axes[1].bar(gender_counts.index.astype(str), gender_counts.values, color="#e45756")
    axes[1].set_title("Genero")
    axes[1].set_xlabel("Codigo")

    icu_col = "ICUType_first"
    if icu_col in processed.columns:
        icu_counts = processed[icu_col].fillna(-1).value_counts().sort_index()
        axes[2].bar(icu_counts.index.astype(str), icu_counts.values, color="#72b7b2")
    axes[2].set_title("Tipo de UCI")
    axes[2].set_xlabel("ICUType")

    fig.tight_layout()
    fig.savefig(figures_dir / "static_distributions_setA.png", dpi=160)
    plt.close(fig)

    label_counts = outcomes["In-hospital_death"].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(["0 (vive)", "1 (fallece)"], label_counts.values, color=["#54a24b", "#e45756"])
    total = label_counts.sum()
    for bar, val in zip(bars, label_counts.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + (total * 0.01),
            f"{val} ({(100 * val / total):.1f}%)",
            ha="center",
            va="bottom",
            fontsize=9,
        )
    ax.set_title("Desbalance de etiqueta en Set A")
    ax.set_ylabel("Numero de estancias")
    fig.tight_layout()
    fig.savefig(figures_dir / "label_imbalance_setA.png", dpi=160)
    plt.close(fig)

    if "HR_mean" in processed.columns:
        hr0 = processed.loc[processed["In-hospital_death"] == 0, "HR_mean"].dropna()
        hr1 = processed.loc[processed["In-hospital_death"] == 1, "HR_mean"].dropna()
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.boxplot([hr0, hr1], tick_labels=["0 (vive)", "1 (fallece)"])
        ax.set_title("HR_mean por etiqueta")
        ax.set_ylabel("HR media en 0-48h")
        fig.tight_layout()
        fig.savefig(figures_dir / "hr_mean_by_outcome_setA.png", dpi=160)
        plt.close(fig)


print("Librerias, rutas y funciones auxiliares listas (todo definido dentro del notebook).")


Librerias, rutas y funciones auxiliares listas (todo definido dentro del notebook).


### Resumen de la seccion anterior
- Se importaron librerias de analisis, visualizacion y manejo de rutas para ejecutar el pipeline de extremo a extremo.
- Se fijaron semillas y directorios de trabajo para garantizar reproducibilidad en carga, procesamiento y export de resultados.
- Se definieron en el propio notebook las funciones auxiliares de lectura, validacion, construccion de features y generacion de figuras.


## 1) Preparación de datos y reutilización del dataset procesado


In [3]:
print("Repo scan breve (archivos clave):")
key_items = [
    ROOT / "HITO3_MIMIC_TRIAGE.ipynb",
    ROOT / "HITO3_TEXT.md",
    CSVS_DIR,
    SET_A_DIR,
    OUTCOMES_A_PATH,
    FIGURES_DIR,
]

for item in key_items:
    print(f"- {item} | existe={item.exists()}")

set_a_files = sorted(SET_A_DIR.glob("*.txt"), key=lambda p: int(p.stem))
print(f"Numero de estancias en set-a: {len(set_a_files)}")
print(f"Primeros RecordID en set-a: {[int(p.stem) for p in set_a_files[:5]]}")


Repo scan breve (archivos clave):
- C:\Users\Natalia\Proyecto-DISIA\Hito 3\HITO3_MIMIC_TRIAGE.ipynb | existe=True
- C:\Users\Natalia\Proyecto-DISIA\Hito 3\HITO3_TEXT.md | existe=True
- C:\Users\Natalia\Proyecto-DISIA\Hito 2\csvs | existe=True
- C:\Users\Natalia\Proyecto-DISIA\Hito 2\predicting-mortality-of-icu-patients-the-physionet-computing-in-cardiology-challenge-2012-1.0.0\set-a | existe=True
- C:\Users\Natalia\Proyecto-DISIA\Hito 2\predicting-mortality-of-icu-patients-the-physionet-computing-in-cardiology-challenge-2012-1.0.0\Outcomes-a.txt | existe=True
- C:\Users\Natalia\Proyecto-DISIA\Hito 3\figures | existe=True
Numero de estancias en set-a: 4000
Primeros RecordID en set-a: [132539, 132540, 132541, 132543, 132545]


### Resumen de la seccion anterior
- Se localizaron y verificaron los activos clave del repo para Hito 2 (contexto, plantilla, referencias, datos RAW y carpeta de CSVs).
- Se confirmo que `set-a` contiene estancias individuales en formato largo (`Time,Parameter,Value`) y se listaron IDs iniciales para validacion de carga.
- Se establecio la base de trabajo: combinar RAW de `set-a` con `Outcomes-a.txt` para construir una tabla por estancia.


## 2) Inspeccion estructural


In [4]:
sample_raw = pd.read_csv(set_a_files[0])
outcomes_a = pd.read_csv(OUTCOMES_A_PATH)

print("Muestra RAW (archivo de una estancia):")
display(sample_raw.head(12))

print("Muestra Outcomes-a:")
display(outcomes_a.head(10))

raw_frames = [read_set_file(path) for path in set_a_files]
raw_all = pd.concat(raw_frames, ignore_index=True)
raw_48h = raw_all.loc[raw_all["Hours"] <= 48].copy()

file_ids = pd.Index([int(p.stem) for p in set_a_files], name="RecordID")
outcome_ids = pd.Index(outcomes_a["RecordID"].astype(int), name="RecordID")

audit = pd.Series(
    {
        "set-a_files": len(set_a_files),
        "outcomes_a_rows": len(outcomes_a),
        "ids_set_a_not_in_outcomes": len(file_ids.difference(outcome_ids)),
        "ids_outcomes_not_in_set_a": len(outcome_ids.difference(file_ids)),
        "rows_raw_total": len(raw_all),
        "rows_raw_0_48h": len(raw_48h),
        "max_hour_raw": float(raw_all["Hours"].max()),
        "n_unique_parameters": raw_all["Parameter"].nunique(),
    }
)
display(audit.to_frame("valor"))


Muestra RAW (archivo de una estancia):


,Time,Parameter,Value
0,00:00,RecordID,132539.00
1,00:00,Age,54.00
2,00:00,Gender,0.00
3,00:00,Height,-1.00
4,00:00,ICUType,4.00
5,00:00,Weight,-1.00
6,00:07,GCS,15.00
7,00:07,HR,73.00
8,00:07,NIDiasABP,65.00
9,00:07,NIMAP,92.33


Muestra Outcomes-a:


,RecordID,SAPS-I,SOFA,Length_of_stay,Survival,In-hospital_death
0,132539,6,1,5,-1,0
1,132540,16,8,8,-1,0
2,132541,21,11,19,-1,0
3,132543,7,1,9,575,0
4,132545,17,2,4,918,0
5,132547,14,11,6,1637,0
6,132548,14,4,9,-1,0
7,132551,19,8,6,5,1
8,132554,11,0,17,38,0
9,132555,14,6,8,-1,0


,valor
set-a_files,4000.0
outcomes_a_rows,4000.0
ids_set_a_not_in_outcomes,0.0
ids_outcomes_not_in_set_a,0.0
rows_raw_total,1757980.0
rows_raw_0_48h,1757980.0
max_hour_raw,48.0
n_unique_parameters,42.0


### Resumen de la seccion anterior
- Se comprobo la estructura del dataset con evidencia: un archivo por estancia (`RecordID`) y etiqueta en `Outcomes-a.txt`.
- Se verifico correspondencia 1:1 entre estancias de `set-a` y filas de outcomes para Set A.
- Se aplico y audito la ventana temporal `<=48h` como restriccion explicita de construccion de features.


In [5]:
static_params = ["RecordID", "Age", "Gender", "Height", "ICUType", "Weight"]
param_stays = (
    raw_48h.groupby("Parameter")["RecordID"]
    .nunique()
    .sort_values(ascending=False)
    .rename("n_estancias")
)
param_table = param_stays.to_frame().reset_index().rename(columns={"index": "Parameter"})
param_table["presencia_pct"] = 100 * param_table["n_estancias"] / len(set_a_files)

units_map = {
    "Age": "years",
    "Weight": "kg",
    "Height": "cm",
    "HR": "bpm",
    "RespRate": "breaths/min",
    "Temp": "C",
    "SysABP": "mmHg",
    "DiasABP": "mmHg",
    "MAP": "mmHg",
    "NISysABP": "mmHg",
    "NIDiasABP": "mmHg",
    "NIMAP": "mmHg",
    "PaO2": "mmHg",
    "PaCO2": "mmHg",
    "FiO2": "fraction",
    "Urine": "mL",
    "WBC": "K/uL",
    "Glucose": "mg/dL",
    "Creatinine": "mg/dL",
}
param_table["unidad_referencia"] = param_table["Parameter"].map(units_map).fillna("ver docs challenge")

print("Variables detectadas (top 20 por presencia en estancias):")
display(param_table.head(20))
print(f"Variables estaticas esperadas: {static_params}")


Variables detectadas (top 20 por presencia en estancias):


,Parameter,n_estancias,presencia_pct,unidad_referencia
0,Age,4000,100.000,years
1,Gender,4000,100.000,ver docs challenge
2,RecordID,4000,100.000,ver docs challenge
3,ICUType,4000,100.000,ver docs challenge
4,Height,4000,100.000,cm
5,Weight,4000,100.000,kg
6,HR,3937,98.425,bpm
7,HCT,3936,98.400,ver docs challenge
8,Creatinine,3936,98.400,mg/dL
9,BUN,3936,98.400,ver docs challenge


Variables estaticas esperadas: ['RecordID', 'Age', 'Gender', 'Height', 'ICUType', 'Weight']


### Resumen de la seccion anterior
- Se inspecciono la cobertura de variables por estancia y se construyo una tabla de presencia para priorizar variables con mayor soporte de datos.
- Se anadieron unidades de referencia para las variables clinicas principales, facilitando interpretacion y control de calidad.
- Este bloque deja definida la estructura de variables estaticas y temporales que alimenta el procesamiento posterior.


## 3) Distribuciones cobertura temporal y missingness


In [6]:
csv_validation = validate_existing_csvs(CSVS_DIR)
print("Validacion de CSVs existentes en ./csvs (antes de reutilizar):")
display(csv_validation)

processed_48h = build_processed_features(raw_48h=raw_48h, outcomes=outcomes_a)
processed_path = CSVS_DIR / "processed_features_48h_setA.csv"
processed_48h.to_csv(processed_path, index=False)

export_figures(raw_48h=raw_48h, processed=processed_48h, outcomes=outcomes_a, figures_dir=FIGURES_DIR)

print(f"Dataset procesado guardado en: {processed_path}")
print(f"Forma final: {processed_48h.shape}")
print(f"Tasa de mortalidad Set A: {100 * processed_48h['In-hospital_death'].mean():.2f}%")


Validacion de CSVs existentes en ./csvs (antes de reutilizar):


""


Dataset procesado guardado en: C:\Users\Natalia\Proyecto-DISIA\Hito 2\csvs\processed_features_48h_setA.csv
Forma final: (4000, 338)
Tasa de mortalidad Set A: 13.85%


### Resumen de la seccion anterior
- Se validaron los CSVs existentes en `./csvs/` antes de reutilizarlos como entrada de modelado.
- Se detecto que los CSVs heredados son tablas resumen y no una matriz por estancia 0-48h, ademas de posibles campos con riesgo de leakage.
- Por trazabilidad y coherencia metodologica, el dataset para Hito 3 se reconstruyo desde RAW y se guardo en `processed_features_48h_setA.csv`.


In [7]:
flag_cols = [c for c in processed_48h.columns if c.endswith("_flag_medido")]
missingness = pd.DataFrame(
    {
        "Parameter": [c.replace("_flag_medido", "") for c in flag_cols],
        "presencia_pct": [100 * processed_48h[c].mean() for c in flag_cols],
    }
).sort_values("presencia_pct")
missingness["missing_pct"] = 100 - missingness["presencia_pct"]

display(missingness.head(15))
display(missingness.tail(15))

figures_created = sorted([p.name for p in FIGURES_DIR.glob("*.png")])
print("Figuras exportadas en reports/figures:")
for f in figures_created:
    print("-", f)


,Parameter,presencia_pct,missing_pct
36,TroponinI,5.125,94.875
7,Cholesterol,7.625,92.375
37,TroponinT,21.575,78.425
32,RespRate,27.525,72.475
4,Albumin,40.375,59.625
0,ALP,42.250,57.750
6,Bilirubin,42.950,57.050
1,ALT,43.025,56.975
2,AST,43.125,56.875
33,SaO2,44.800,55.200


,Parameter,presencia_pct,missing_pct
27,Na,98.125,1.875
39,WBC,98.175,1.825
30,Platelets,98.300,1.700
5,BUN,98.400,1.600
11,GCS,98.400,1.600
35,Temp,98.400,1.600
8,Creatinine,98.400,1.600
15,HCT,98.400,1.600
16,HR,98.425,1.575
3,Age,100.000,0.000


Figuras exportadas en reports/figures:
- correlation_subset_setA_0_48h.png
- hr_mean_by_outcome_setA.png
- label_imbalance_setA.png
- measurement_volume_top20_setA_0_48h.png
- missingness_setA_0_48h.png
- rf_tuned_feature_importance_top.png
- rf_tuned_shap_mean_abs_top.png
- static_distributions_setA.png
- temporal_coverage_setA_0_48h.png


### Resumen de la seccion anterior
- Se cuantifico missingness por variable a partir de `flag_medido`, permitiendo identificar variables con alta y baja cobertura.
- Se verifico el conjunto de figuras exportadas para documentar cobertura temporal, distribuciones estaticas y desbalance de etiqueta.
- Este resumen conecta la exploracion descriptiva con decisiones de seleccion y tratamiento de variables para modelado.


## 4) Correlaciones y análisis por tipo de UCI


In [8]:
mean_cols = [c for c in processed_48h.columns if c.endswith("_mean")]
selected = [c for c in ["HR_mean", "MAP_mean", "SysABP_mean", "RespRate_mean", "Temp_mean", "Creatinine_mean", "WBC_mean"] if c in mean_cols]

if len(selected) >= 2:
    corr = processed_48h[selected].corr()
    fig, ax = plt.subplots(figsize=(7, 5))
    img = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(selected)))
    ax.set_yticks(range(len(selected)))
    ax.set_xticklabels(selected, rotation=45, ha="right")
    ax.set_yticklabels(selected)
    ax.set_title("Correlaciones (subset de variables medias 0-48h)")
    fig.colorbar(img, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    corr_path = FIGURES_DIR / "correlation_subset_setA_0_48h.png"
    fig.savefig(corr_path, dpi=160)
    plt.close(fig)
    display(corr.round(2))
    print(f"Figura guardada: {corr_path}")
else:
    print("No hay suficientes columnas para matriz de correlacion.")


,HR_mean,MAP_mean,SysABP_mean,RespRate_mean,Temp_mean,Creatinine_mean,WBC_mean
HR_mean,1.00,0.01,-0.19,0.33,0.21,-0.03,0.12
MAP_mean,0.01,1.00,0.38,0.00,0.03,-0.06,-0.06
SysABP_mean,-0.19,0.38,1.00,-0.10,0.02,-0.04,-0.10
RespRate_mean,0.33,0.00,-0.10,1.00,0.08,-0.03,0.15
Temp_mean,0.21,0.03,0.02,0.08,1.00,-0.11,0.04
Creatinine_mean,-0.03,-0.06,-0.04,-0.03,-0.11,1.00,0.03
WBC_mean,0.12,-0.06,-0.10,0.15,0.04,0.03,1.00


Figura guardada: C:\Users\Natalia\Proyecto-DISIA\Hito 3\figures\correlation_subset_setA_0_48h.png


### Resumen de la seccion anterior
- Se calcularon correlaciones en un subconjunto de variables medias 0-48h para detectar relaciones lineales relevantes.
- Se genero y guardo una figura de correlacion para soporte del informe y trazabilidad del analisis.
- El resultado orienta controles de colinealidad y la eleccion de baselines lineales/no lineales en fases posteriores.


### Analisis por tipo de UCI
Adaptacion del bloque "analisis por posicion" de la plantilla: aqui segmentamos por `ICUType` para evaluar heterogeneidad clinica.


In [9]:
if "ICUType_first" in processed_48h.columns:
    cohort = (
        processed_48h.groupby("ICUType_first", dropna=False)
        .agg(
            n_estancias=("RecordID", "count"),
            mortalidad_pct=("In-hospital_death", lambda s: 100 * s.mean()),
            edad_media=("Age_first", "mean"),
            hr_media=("HR_mean", "mean"),
        )
        .sort_values("n_estancias", ascending=False)
    )
    display(cohort.round(2))
else:
    print("ICUType_first no encontrado.")


,n_estancias,mortalidad_pct,edad_media,hr_media
ICUType_first,,,,
3.0,1481,18.57,63.05,89.78
4.0,1068,14.51,60.27,86.43
2.0,874,4.92,67.80,86.28
1.0,577,14.04,69.31,83.27


### Resumen de la seccion anterior
- Se segmentaron las estancias por `ICUType` para comparar volumen, mortalidad y perfiles clinicos agregados.
- El analisis evidencia heterogeneidad entre subcohortes, util para interpretar diferencias de riesgo base.
- Esta vista estratificada ayuda a definir validaciones y analisis de fairness/robustez en Hito 3.


## 5) Baseline experimental

A partir de este punto, el análisis deja de centrarse en la construcción del dataset y pasa a la fase de modelado, experimentación y validación.

En coherencia con el Hito 2, se utiliza como entrada el dataset tabular por estancia `processed_features_48h_setA.csv`, ya generado a partir de los datos raw de Set A en la ventana temporal 0–48h. Sobre este dataset se define un baseline reproducible de scoring probabilístico de mortalidad intrahospitalaria, cuya salida se emplea además para construir un ranking clínico de priorización.

Este baseline servirá como punto de referencia para la comparación posterior de modelos más avanzados en el Hito 3.

### Definición del baseline

Como primer experimento se construye un baseline basado en regresión logística. La elección de este modelo responde a varios motivos: 
1. proporciona una referencia interpretable y ampliamente utilizada en problemas de riesgo clínico;
2. resulta adecuada como primera aproximación sobre datos tabulares agregados;
3. permite obtener probabilidades continuas de mortalidad, compatibles tanto con métricas probabilísticas como con métricas de ranking clínico.

En esta primera versión, el baseline se entrena sobre un subconjunto reducido de variables clínicas y demográficas seleccionadas manualmente, con imputación por mediana y estandarización calculadas exclusivamente sobre entrenamiento.

### Implementación manual inicial del baseline (solo trazabilidad)

> Nota: este bloque corresponde a la primera implementación manual del baseline. Se conserva únicamente por trazabilidad del desarrollo, pero la referencia oficial para el Hito 3 pasa a ser la versión equivalente implementada con `scikit-learn`.

In [10]:
candidate_features = [
    "Age_first",
    "Gender_first",
    "ICUType_first",
    "GCS_mean",
    "HR_mean",
    "MAP_mean",
    "SysABP_mean",
    "DiasABP_mean",
    "RespRate_mean",
    "Temp_mean",
    "Creatinine_mean",
    "BUN_mean",
    "WBC_mean",
    "Urine_mean",
]
features = [f for f in candidate_features if f in processed_48h.columns]
print(f"Features disponibles para baseline: {features}")

model_df = processed_48h[["RecordID", "In-hospital_death"] + features].copy()
for c in features:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce")

is_test = (model_df["RecordID"] % 5 == 0)
train_df = model_df.loc[~is_test].copy()
test_df = model_df.loc[is_test].copy()

medians = train_df[features].median()
X_train = train_df[features].fillna(medians).to_numpy(dtype=float)
X_test = test_df[features].fillna(medians).to_numpy(dtype=float)
y_train = train_df["In-hospital_death"].to_numpy(dtype=float)
y_test = test_df["In-hospital_death"].to_numpy(dtype=int)

mu = X_train.mean(axis=0)
sigma = X_train.std(axis=0)
sigma[sigma == 0] = 1.0
X_train = (X_train - mu) / sigma
X_test = (X_test - mu) / sigma

def sigmoid(z):
    z = np.clip(z, -35, 35)
    return 1.0 / (1.0 + np.exp(-z))

def fit_logreg_gd(X, y, lr=0.05, n_iter=4000, l2=1e-4):
    n, m = X.shape
    w = np.zeros(m)
    b = 0.0
    for _ in range(n_iter):
        p = sigmoid(X @ w + b)
        err = p - y
        grad_w = (X.T @ err) / n + l2 * w
        grad_b = err.mean()
        w -= lr * grad_w
        b -= lr * grad_b
    return w, b

w, b = fit_logreg_gd(X_train, y_train)
test_proba = sigmoid(X_test @ w + b)

ranking_test = test_df[["RecordID", "In-hospital_death"]].copy()
ranking_test["pred_proba_mortality"] = test_proba
ranking_test = ranking_test.sort_values("pred_proba_mortality", ascending=False).reset_index(drop=True)
display(ranking_test.head(20))


Features disponibles para baseline: ['Age_first', 'Gender_first', 'ICUType_first', 'GCS_mean', 'HR_mean', 'MAP_mean', 'SysABP_mean', 'DiasABP_mean', 'RespRate_mean', 'Temp_mean', 'Creatinine_mean', 'BUN_mean', 'WBC_mean', 'Urine_mean']


,RecordID,In-hospital_death,pred_proba_mortality
0,134580,1,0.878449
1,137325,1,0.874071
2,132895,0,0.821284
3,138505,0,0.783028
4,142160,1,0.769529
5,133225,0,0.690961
6,139820,0,0.688423
7,142570,1,0.682155
8,134945,1,0.679730
9,138530,1,0.672373


### Resumen de la seccion anterior
- Se entreno un baseline de scoring probabilistico con variables agregadas 0-48h y particion reproducible por `RecordID`.
- Se estimaron probabilidades de mortalidad por estancia y se construyo el ranking descendente de prioridad clinica.
- Este bloque formaliza la transicion de datos procesados a decision operativa basada en riesgo.


### Reimplementación oficial del baseline con scikit-learn

Para facilitar la comparación posterior entre modelos y alinearse con prácticas estándar de experimentación en machine learning, el baseline anterior se reimplementa utilizando `scikit-learn`.

Esta nueva versión mantiene la misma lógica experimental: mismo subconjunto de variables, misma partición entrenamiento/test y mismo objetivo de scoring probabilístico. El cambio afecta únicamente a la implementación técnica del modelo y del preprocesado.

In [11]:
# Baseline equivalente con scikit-learn

features = [f for f in candidate_features if f in processed_48h.columns]
print(f"Features disponibles para baseline sklearn: {features}")

model_df = processed_48h[["RecordID", "In-hospital_death"] + features].copy()
for c in features:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce")

is_test = (model_df["RecordID"] % 5 == 0)
train_df_sk = model_df.loc[~is_test].copy()
test_df_sk = model_df.loc[is_test].copy()

X_train_sk = train_df_sk[features]
X_test_sk = test_df_sk[features]
y_train_sk = train_df_sk["In-hospital_death"].astype(int)
y_test_sk = test_df_sk["In-hospital_death"].astype(int)

baseline_sk = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_SEED))
])

baseline_sk.fit(X_train_sk, y_train_sk)
test_proba_sk = baseline_sk.predict_proba(X_test_sk)[:, 1]

ranking_test_sk = test_df_sk[["RecordID", "In-hospital_death"]].copy()
ranking_test_sk["pred_proba_mortality"] = test_proba_sk
ranking_test_sk = ranking_test_sk.sort_values("pred_proba_mortality", ascending=False).reset_index(drop=True)

display(ranking_test_sk.head(20))

Features disponibles para baseline sklearn: ['Age_first', 'Gender_first', 'ICUType_first', 'GCS_mean', 'HR_mean', 'MAP_mean', 'SysABP_mean', 'DiasABP_mean', 'RespRate_mean', 'Temp_mean', 'Creatinine_mean', 'BUN_mean', 'WBC_mean', 'Urine_mean']


,RecordID,In-hospital_death,pred_proba_mortality
0,134580,1,0.877533
1,137325,1,0.873151
2,132895,0,0.820182
3,138505,0,0.782195
4,142160,1,0.768732
5,133225,0,0.690410
6,139820,0,0.687652
7,142570,1,0.680800
8,134945,1,0.678818
9,138530,1,0.671649


### Métricas de ranking clínico (Recall@k y porcentaje de fallecidos en top-k)


In [12]:
def ranking_metrics(y_true, y_score, ks=(25, 50, 100, 200)):
    order = np.argsort(-y_score)
    y_sorted = y_true[order]
    total_pos = max(y_true.sum(), 1)
    rows = []
    for k in ks:
        k = min(k, len(y_sorted))
        top_k = y_sorted[:k]
        deaths_top_k = int(top_k.sum())
        rows.append(
            {
                "k": k,
                "deaths_top_k": deaths_top_k,
                "Recall_at_k": deaths_top_k / total_pos,
                "pct_fallecidos_en_top_k": deaths_top_k / k,
            }
        )
    return pd.DataFrame(rows)

rank_metrics = ranking_metrics(y_true=y_test, y_score=test_proba, ks=(25, 50, 100, 200))
display(rank_metrics)


,k,deaths_top_k,Recall_at_k,pct_fallecidos_en_top_k
0,25,16,0.168421,0.640
1,50,23,0.242105,0.460
2,100,43,0.452632,0.430
3,200,61,0.642105,0.305


In [13]:
rank_metrics_sk = ranking_metrics(y_true=y_test_sk.to_numpy(), y_score=test_proba_sk, ks=(25, 50, 100, 200))
display(rank_metrics_sk)

auc_sk = roc_auc_score(y_test_sk, test_proba_sk)
brier_sk = brier_score_loss(y_test_sk, test_proba_sk)

prob_metrics_sk = pd.DataFrame(
    {
        "metrica": ["AUC_ROC", "Brier_score", "prevalencia_test"],
        "valor": [auc_sk, brier_sk, float(np.mean(y_test_sk))],
    }
)
display(prob_metrics_sk)

print("Train shape sklearn:", train_df_sk.shape)
print("Test shape sklearn:", test_df_sk.shape)

,k,deaths_top_k,Recall_at_k,pct_fallecidos_en_top_k
0,25,16,0.168421,0.640
1,50,23,0.242105,0.460
2,100,43,0.452632,0.430
3,200,61,0.642105,0.305


,metrica,valor
0,AUC_ROC,0.780608
1,Brier_score,0.088736
2,prevalencia_test,0.118012


Train shape sklearn: (3195, 16)
Test shape sklearn: (805, 16)


### Comparación entre implementación manual y versión con scikit-learn

La reimplementación del baseline con `scikit-learn` reproduce esencialmente los mismos resultados que la versión manual anterior, tanto en métricas probabilísticas como en métricas de ranking clínico. Esto confirma que el comportamiento observado no depende de una implementación ad hoc concreta, sino de la señal contenida en los datos y de la propia formulación del modelo baseline.

A partir de este punto, la versión basada en `scikit-learn` se adopta como referencia principal para el resto del Hito 3, por su mayor claridad metodológica, reproducibilidad y facilidad para extender la comparación a modelos adicionales.

### Resumen de la seccion anterior
- Se calcularon metricas top-k orientadas a priorizacion (Recall@k y porcentaje de fallecidos en top-k).
- El analisis mide de forma directa cuanto riesgo real concentra el tramo alto del ranking.
- Estas metricas complementan las probabilisticas para evaluar utilidad clinica en escenarios de recursos limitados.


### Métricas probabilísticas del baseline manual

In [14]:
def auc_roc_rank(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    pos = y_true == 1
    neg = y_true == 0
    n_pos = pos.sum()
    n_neg = neg.sum()
    if n_pos == 0 or n_neg == 0:
        return np.nan
    ranks = pd.Series(y_score).rank(method="average").to_numpy()
    auc = (ranks[pos].sum() - (n_pos * (n_pos + 1) / 2)) / (n_pos * n_neg)
    return float(auc)

auc = auc_roc_rank(y_test, test_proba)
brier = float(np.mean((test_proba - y_test) ** 2))

prob_metrics = pd.DataFrame(
    {
        "metrica": ["AUC_ROC", "Brier_score", "prevalencia_test"],
        "valor": [auc, brier, float(np.mean(y_test))],
    }
)
display(prob_metrics)


,metrica,valor
0,AUC_ROC,0.780563
1,Brier_score,0.088748
2,prevalencia_test,0.118012


In [15]:
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nRanking metrics:")
display(rank_metrics)

print("\nProbabilistic metrics:")
display(prob_metrics)

Train shape: (3195, 16)
Test shape: (805, 16)

Ranking metrics:


,k,deaths_top_k,Recall_at_k,pct_fallecidos_en_top_k
0,25,16,0.168421,0.640
1,50,23,0.242105,0.460
2,100,43,0.452632,0.430
3,200,61,0.642105,0.305



Probabilistic metrics:


,metrica,valor
0,AUC_ROC,0.780563
1,Brier_score,0.088748
2,prevalencia_test,0.118012


### Interpretación inicial del baseline

El baseline de regresión logística obtiene un AUC-ROC de 0.7806 y un Brier score de 0.0887 sobre el conjunto de test. Estos resultados indican que, incluso con un subconjunto reducido de variables clínicas y demográficas agregadas, el sistema ya es capaz de separar de forma útil pacientes con distinto nivel de riesgo.

Desde el punto de vista operativo, el ranking concentra una proporción de fallecidos muy superior a la prevalencia base del conjunto de test. En particular, el top-25 contiene un 64% de pacientes fallecidos y el top-100 un 43%, lo que respalda la utilidad del enfoque para tareas de priorización clínica.

### Cierre de la fase de baseline

Hasta este punto se ha verificado la estructura del dataset, se ha reconstruido el dataset procesado por estancia y se ha establecido un baseline reproducible de scoring probabilístico. A partir de aquí, el análisis se centra en la comparación y validación de modelos alternativos.

## 6) Comparación inicial de modelos

Como primer modelo alternativo al baseline lineal, se evalúa un clasificador Random Forest. Este modelo resulta adecuado para datos tabulares clínicos, permite capturar relaciones no lineales e interacciones entre variables, y constituye una referencia habitual en problemas de predicción de riesgo.

En esta primera comparación se mantiene exactamente la misma partición entrenamiento/test y el mismo subconjunto de variables utilizado en el baseline, de modo que cualquier diferencia observada pueda atribuirse al modelo y no a cambios en el protocolo experimental.

In [16]:
# Random Forest con la misma configuracion experimental del baseline

features = [f for f in candidate_features if f in processed_48h.columns]
print(f"Features disponibles para Random Forest: {features}")

model_df = processed_48h[["RecordID", "In-hospital_death"] + features].copy()
for c in features:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce")

is_test = (model_df["RecordID"] % 5 == 0)
train_df_rf = model_df.loc[~is_test].copy()
test_df_rf = model_df.loc[is_test].copy()

X_train_rf = train_df_rf[features]
X_test_rf = test_df_rf[features]
y_train_rf = train_df_rf["In-hospital_death"].astype(int)
y_test_rf = test_df_rf["In-hospital_death"].astype(int)

rf_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        random_state=RANDOM_SEED,
        n_jobs=-1
    ))
])

rf_model.fit(X_train_rf, y_train_rf)
test_proba_rf = rf_model.predict_proba(X_test_rf)[:, 1]

ranking_test_rf = test_df_rf[["RecordID", "In-hospital_death"]].copy()
ranking_test_rf["pred_proba_mortality"] = test_proba_rf
ranking_test_rf = ranking_test_rf.sort_values("pred_proba_mortality", ascending=False).reset_index(drop=True)

display(ranking_test_rf.head(20))

Features disponibles para Random Forest: ['Age_first', 'Gender_first', 'ICUType_first', 'GCS_mean', 'HR_mean', 'MAP_mean', 'SysABP_mean', 'DiasABP_mean', 'RespRate_mean', 'Temp_mean', 'Creatinine_mean', 'BUN_mean', 'WBC_mean', 'Urine_mean']


,RecordID,In-hospital_death,pred_proba_mortality
0,132960,1,0.776310
1,134580,1,0.702667
2,142160,1,0.701278
3,137325,1,0.678706
4,133780,1,0.648635
5,134835,1,0.637317
6,138530,1,0.626389
7,135305,1,0.606714
8,134080,1,0.602176
9,133975,0,0.601389


In [17]:
rank_metrics_rf = ranking_metrics(y_true=y_test_rf.to_numpy(), y_score=test_proba_rf, ks=(25, 50, 100, 200))
display(rank_metrics_rf)

auc_rf = roc_auc_score(y_test_rf, test_proba_rf)
brier_rf = brier_score_loss(y_test_rf, test_proba_rf)

prob_metrics_rf = pd.DataFrame(
    {
        "metrica": ["AUC_ROC", "Brier_score", "prevalencia_test"],
        "valor": [auc_rf, brier_rf, float(np.mean(y_test_rf))],
    }
)
display(prob_metrics_rf)

print("Train shape RF:", train_df_rf.shape)
print("Test shape RF:", test_df_rf.shape)

,k,deaths_top_k,Recall_at_k,pct_fallecidos_en_top_k
0,25,17,0.178947,0.68
1,50,28,0.294737,0.56
2,100,44,0.463158,0.44
3,200,64,0.673684,0.32


,metrica,valor
0,AUC_ROC,0.851987
1,Brier_score,0.081791
2,prevalencia_test,0.118012


Train shape RF: (3195, 16)
Test shape RF: (805, 16)


In [18]:
comparison_prob = pd.DataFrame([
    {"modelo": "Logistic Regression (baseline)", "AUC_ROC": auc_sk, "Brier_score": brier_sk},
    {"modelo": "Random Forest", "AUC_ROC": auc_rf, "Brier_score": brier_rf},
])

comparison_rank = pd.DataFrame([
    {
        "modelo": "Logistic Regression (baseline)",
        "Recall@25": rank_metrics_sk.loc[rank_metrics_sk["k"] == 25, "Recall_at_k"].iloc[0],
        "Recall@50": rank_metrics_sk.loc[rank_metrics_sk["k"] == 50, "Recall_at_k"].iloc[0],
        "Recall@100": rank_metrics_sk.loc[rank_metrics_sk["k"] == 100, "Recall_at_k"].iloc[0],
        "% fallecidos top25": rank_metrics_sk.loc[rank_metrics_sk["k"] == 25, "pct_fallecidos_en_top_k"].iloc[0],
    },
    {
        "modelo": "Random Forest",
        "Recall@25": rank_metrics_rf.loc[rank_metrics_rf["k"] == 25, "Recall_at_k"].iloc[0],
        "Recall@50": rank_metrics_rf.loc[rank_metrics_rf["k"] == 50, "Recall_at_k"].iloc[0],
        "Recall@100": rank_metrics_rf.loc[rank_metrics_rf["k"] == 100, "Recall_at_k"].iloc[0],
        "% fallecidos top25": rank_metrics_rf.loc[rank_metrics_rf["k"] == 25, "pct_fallecidos_en_top_k"].iloc[0],
    },
])

print("Comparacion de metricas probabilisticas:")
display(comparison_prob)

print("Comparacion de metricas de ranking:")
display(comparison_rank)

Comparacion de metricas probabilisticas:


,modelo,AUC_ROC,Brier_score
0,Logistic Regression (baseline),0.780608,0.088736
1,Random Forest,0.851987,0.081791


Comparacion de metricas de ranking:


,modelo,Recall@25,Recall@50,Recall@100,% fallecidos top25
0,Logistic Regression (baseline),0.168421,0.242105,0.452632,0.64
1,Random Forest,0.178947,0.294737,0.463158,0.68


### Interpretación inicial de la comparación baseline vs. Random Forest

El modelo Random Forest mejora de forma consistente al baseline de regresión logística tanto en métricas probabilísticas como en métricas de ranking clínico. En particular, el aumento del AUC-ROC y la reducción del Brier score sugieren una mejor capacidad de discriminación y una estimación probabilística más precisa.

Desde la perspectiva operativa, también se observa una mayor concentración de pacientes fallecidos en la parte alta del ranking. Esto indica que introducir no linealidad e interacciones entre variables clínicas aporta valor en el problema de priorización de pacientes en UCI.

In [19]:
model_summary = pd.DataFrame([
    {
        "modelo": "Logistic Regression",
        "AUC_ROC": auc_sk,
        "Brier_score": brier_sk,
        "Recall@25": rank_metrics_sk.loc[rank_metrics_sk["k"] == 25, "Recall_at_k"].iloc[0],
        "Recall@50": rank_metrics_sk.loc[rank_metrics_sk["k"] == 50, "Recall_at_k"].iloc[0],
        "Recall@100": rank_metrics_sk.loc[rank_metrics_sk["k"] == 100, "Recall_at_k"].iloc[0],
        "% fallecidos top25": rank_metrics_sk.loc[rank_metrics_sk["k"] == 25, "pct_fallecidos_en_top_k"].iloc[0],
    },
    {
        "modelo": "Random Forest",
        "AUC_ROC": auc_rf,
        "Brier_score": brier_rf,
        "Recall@25": rank_metrics_rf.loc[rank_metrics_rf["k"] == 25, "Recall_at_k"].iloc[0],
        "Recall@50": rank_metrics_rf.loc[rank_metrics_rf["k"] == 50, "Recall_at_k"].iloc[0],
        "Recall@100": rank_metrics_rf.loc[rank_metrics_rf["k"] == 100, "Recall_at_k"].iloc[0],
        "% fallecidos top25": rank_metrics_rf.loc[rank_metrics_rf["k"] == 25, "pct_fallecidos_en_top_k"].iloc[0],
    },
])

display(model_summary.round(4))

,modelo,AUC_ROC,Brier_score,Recall@25,Recall@50,Recall@100,% fallecidos top25
0,Logistic Regression,0.7806,0.0887,0.1684,0.2421,0.4526,0.64
1,Random Forest,0.8520,0.0818,0.1789,0.2947,0.4632,0.68


## Tercer modelo candidato: Gradient Boosting

Como siguiente modelo candidato se evalúa un enfoque de boosting basado en árboles. Frente a Random Forest, los modelos de boosting construyen secuencialmente árboles débiles que corrigen errores previos, lo que a menudo permite obtener mejor capacidad predictiva en datos tabulares.

En esta primera comparación se mantiene el mismo subconjunto de variables y la misma partición entrenamiento/test, con el fin de comparar los modelos en condiciones homogéneas.

In [20]:
# HistGradientBoosting con la misma configuracion experimental del baseline

features = [f for f in candidate_features if f in processed_48h.columns]
print(f"Features disponibles para Gradient Boosting: {features}")

model_df = processed_48h[["RecordID", "In-hospital_death"] + features].copy()
for c in features:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce")

is_test = (model_df["RecordID"] % 5 == 0)
train_df_gb = model_df.loc[~is_test].copy()
test_df_gb = model_df.loc[is_test].copy()

X_train_gb = train_df_gb[features]
X_test_gb = test_df_gb[features]
y_train_gb = train_df_gb["In-hospital_death"].astype(int)
y_test_gb = test_df_gb["In-hospital_death"].astype(int)

gb_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_depth=4,
        min_samples_leaf=20,
        random_state=RANDOM_SEED
    ))
])

gb_model.fit(X_train_gb, y_train_gb)
test_proba_gb = gb_model.predict_proba(X_test_gb)[:, 1]

ranking_test_gb = test_df_gb[["RecordID", "In-hospital_death"]].copy()
ranking_test_gb["pred_proba_mortality"] = test_proba_gb
ranking_test_gb = ranking_test_gb.sort_values("pred_proba_mortality", ascending=False).reset_index(drop=True)

display(ranking_test_gb.head(20))

Features disponibles para Gradient Boosting: ['Age_first', 'Gender_first', 'ICUType_first', 'GCS_mean', 'HR_mean', 'MAP_mean', 'SysABP_mean', 'DiasABP_mean', 'RespRate_mean', 'Temp_mean', 'Creatinine_mean', 'BUN_mean', 'WBC_mean', 'Urine_mean']


,RecordID,In-hospital_death,pred_proba_mortality
0,134580,1,0.921315
1,134835,1,0.857076
2,132960,1,0.843476
3,133975,0,0.824290
4,141460,0,0.806644
5,138530,1,0.779893
6,139230,1,0.766167
7,135210,1,0.760084
8,136810,1,0.734971
9,140690,0,0.713076


In [21]:
rank_metrics_gb = ranking_metrics(y_true=y_test_gb.to_numpy(), y_score=test_proba_gb, ks=(25, 50, 100, 200))
display(rank_metrics_gb)

auc_gb = roc_auc_score(y_test_gb, test_proba_gb)
brier_gb = brier_score_loss(y_test_gb, test_proba_gb)

prob_metrics_gb = pd.DataFrame(
    {
        "metrica": ["AUC_ROC", "Brier_score", "prevalencia_test"],
        "valor": [auc_gb, brier_gb, float(np.mean(y_test_gb))],
    }
)
display(prob_metrics_gb)

print("Train shape GB:", train_df_gb.shape)
print("Test shape GB:", test_df_gb.shape)

,k,deaths_top_k,Recall_at_k,pct_fallecidos_en_top_k
0,25,15,0.157895,0.600
1,50,28,0.294737,0.560
2,100,44,0.463158,0.440
3,200,63,0.663158,0.315


,metrica,valor
0,AUC_ROC,0.840133
1,Brier_score,0.083897
2,prevalencia_test,0.118012


Train shape GB: (3195, 16)
Test shape GB: (805, 16)


In [22]:
model_summary = pd.DataFrame([
    {
        "modelo": "Logistic Regression",
        "AUC_ROC": auc_sk,
        "Brier_score": brier_sk,
        "Recall@25": rank_metrics_sk.loc[rank_metrics_sk["k"] == 25, "Recall_at_k"].iloc[0],
        "Recall@50": rank_metrics_sk.loc[rank_metrics_sk["k"] == 50, "Recall_at_k"].iloc[0],
        "Recall@100": rank_metrics_sk.loc[rank_metrics_sk["k"] == 100, "Recall_at_k"].iloc[0],
        "% fallecidos top25": rank_metrics_sk.loc[rank_metrics_sk["k"] == 25, "pct_fallecidos_en_top_k"].iloc[0],
    },
    {
        "modelo": "Random Forest",
        "AUC_ROC": auc_rf,
        "Brier_score": brier_rf,
        "Recall@25": rank_metrics_rf.loc[rank_metrics_rf["k"] == 25, "Recall_at_k"].iloc[0],
        "Recall@50": rank_metrics_rf.loc[rank_metrics_rf["k"] == 50, "Recall_at_k"].iloc[0],
        "Recall@100": rank_metrics_rf.loc[rank_metrics_rf["k"] == 100, "Recall_at_k"].iloc[0],
        "% fallecidos top25": rank_metrics_rf.loc[rank_metrics_rf["k"] == 25, "pct_fallecidos_en_top_k"].iloc[0],
    },
    {
        "modelo": "Gradient Boosting",
        "AUC_ROC": auc_gb,
        "Brier_score": brier_gb,
        "Recall@25": rank_metrics_gb.loc[rank_metrics_gb["k"] == 25, "Recall_at_k"].iloc[0],
        "Recall@50": rank_metrics_gb.loc[rank_metrics_gb["k"] == 50, "Recall_at_k"].iloc[0],
        "Recall@100": rank_metrics_gb.loc[rank_metrics_gb["k"] == 100, "Recall_at_k"].iloc[0],
        "% fallecidos top25": rank_metrics_gb.loc[rank_metrics_gb["k"] == 25, "pct_fallecidos_en_top_k"].iloc[0],
    },
])

display(model_summary.round(4))

,modelo,AUC_ROC,Brier_score,Recall@25,Recall@50,Recall@100,% fallecidos top25
0,Logistic Regression,0.7806,0.0887,0.1684,0.2421,0.4526,0.64
1,Random Forest,0.8520,0.0818,0.1789,0.2947,0.4632,0.68
2,Gradient Boosting,0.8401,0.0839,0.1579,0.2947,0.4632,0.60


### Interpretación inicial de la comparación entre modelos

La comparación inicial muestra que los modelos basados en árboles mejoran claramente al baseline lineal, lo que sugiere que el problema presenta relaciones no lineales e interacciones entre variables clínicas que la regresión logística no captura completamente.

Entre los modelos evaluados, Random Forest obtiene el mejor equilibrio global en esta primera fase, al alcanzar el mayor AUC-ROC, el menor Brier score y una mejor concentración de mortalidad en la parte alta del ranking. Aunque Gradient Boosting también mejora al baseline, no supera a Random Forest con la configuración utilizada.

In [23]:
model_summary_sorted = model_summary.sort_values(
    by=["AUC_ROC", "Brier_score"],
    ascending=[False, True]
).reset_index(drop=True)

display(model_summary_sorted.round(4))

,modelo,AUC_ROC,Brier_score,Recall@25,Recall@50,Recall@100,% fallecidos top25
0,Random Forest,0.8520,0.0818,0.1789,0.2947,0.4632,0.68
1,Gradient Boosting,0.8401,0.0839,0.1579,0.2947,0.4632,0.60
2,Logistic Regression,0.7806,0.0887,0.1684,0.2421,0.4526,0.64


### Modelo candidato líder tras la comparación inicial

A la vista de los resultados obtenidos en esta comparación inicial, Random Forest se selecciona provisionalmente como modelo candidato líder para las siguientes fases del Hito 3. Esta decisión es todavía preliminar y deberá confirmarse mediante estrategias de validación más robustas y, en su caso, mediante ajuste de hiperparámetros.

## 7) Validación cruzada estratificada

La comparación anterior se ha realizado sobre una única partición entrenamiento/test, útil como primera aproximación pero insuficiente para extraer conclusiones sólidas sobre la robustez de los modelos. 

Para reforzar la validez experimental, a continuación se evalúan los modelos mediante validación cruzada estratificada. Esta estrategia permite obtener estimaciones más estables del rendimiento al repetir el entrenamiento y evaluación sobre distintas particiones de los datos, preservando la proporción de la clase minoritaria en cada fold.

In [24]:
# Dataset comun para validacion cruzada

features = [f for f in candidate_features if f in processed_48h.columns]
print(f"Features para validacion cruzada: {features}")

cv_df = processed_48h[["RecordID", "In-hospital_death"] + features].copy()
for c in features:
    cv_df[c] = pd.to_numeric(cv_df[c], errors="coerce")

X_cv = cv_df[features]
y_cv = cv_df["In-hospital_death"].astype(int)

print("Shape X_cv:", X_cv.shape)
print("Shape y_cv:", y_cv.shape)
print("Prevalencia global:", float(y_cv.mean()))

Features para validacion cruzada: ['Age_first', 'Gender_first', 'ICUType_first', 'GCS_mean', 'HR_mean', 'MAP_mean', 'SysABP_mean', 'DiasABP_mean', 'RespRate_mean', 'Temp_mean', 'Creatinine_mean', 'BUN_mean', 'WBC_mean', 'Urine_mean']
Shape X_cv: (4000, 14)
Shape y_cv: (4000,)
Prevalencia global: 0.1385


In [25]:
# Modelos oficiales para validacion cruzada

cv_models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_SEED))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=2,
            random_state=RANDOM_SEED,
            n_jobs=-1
        ))
    ]),
    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.05,
            max_depth=4,
            min_samples_leaf=20,
            random_state=RANDOM_SEED
        ))
    ]),
}

list(cv_models.keys())

['Logistic Regression', 'Random Forest', 'Gradient Boosting']

In [26]:
def cross_validate_models(models, X, y, n_splits=5, random_state=RANDOM_SEED):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    rows = []

    for model_name, model in models.items():
        fold_idx = 1

        for train_idx, val_idx in skf.split(X, y):
            X_train_fold = X.iloc[train_idx]
            X_val_fold = X.iloc[val_idx]
            y_train_fold = y.iloc[train_idx]
            y_val_fold = y.iloc[val_idx]

            fitted_model = clone(model)
            fitted_model.fit(X_train_fold, y_train_fold)
            val_proba = fitted_model.predict_proba(X_val_fold)[:, 1]

            auc_fold = roc_auc_score(y_val_fold, val_proba)
            brier_fold = brier_score_loss(y_val_fold, val_proba)

            rows.append({
                "modelo": model_name,
                "fold": fold_idx,
                "AUC_ROC": auc_fold,
                "Brier_score": brier_fold,
                "prevalencia_fold": float(y_val_fold.mean()),
            })

            fold_idx += 1

    return pd.DataFrame(rows)

In [27]:
cv_results = cross_validate_models(cv_models, X_cv, y_cv, n_splits=5, random_state=RANDOM_SEED)
display(cv_results.round(4))

,modelo,fold,AUC_ROC,Brier_score,prevalencia_fold
0,Logistic Regression,1,0.7611,0.1051,0.1375
1,Logistic Regression,2,0.8126,0.0988,0.1388
2,Logistic Regression,3,0.7801,0.1048,0.1388
3,Logistic Regression,4,0.7735,0.1049,0.1388
4,Logistic Regression,5,0.8059,0.0997,0.1388
5,Random Forest,1,0.7874,0.1027,0.1375
6,Random Forest,2,0.8269,0.0950,0.1388
7,Random Forest,3,0.8261,0.0961,0.1388
8,Random Forest,4,0.8216,0.0975,0.1388
9,Random Forest,5,0.8492,0.0902,0.1388


In [28]:
cv_summary = (
    cv_results
    .groupby("modelo")
    .agg(
        AUC_ROC_mean=("AUC_ROC", "mean"),
        AUC_ROC_std=("AUC_ROC", "std"),
        Brier_score_mean=("Brier_score", "mean"),
        Brier_score_std=("Brier_score", "std"),
    )
    .reset_index()
    .sort_values(by=["AUC_ROC_mean", "Brier_score_mean"], ascending=[False, True])
)

display(cv_summary.round(4))

,modelo,AUC_ROC_mean,AUC_ROC_std,Brier_score_mean,Brier_score_std
0,Gradient Boosting,0.8254,0.0205,0.0969,0.0050
2,Random Forest,0.8222,0.0222,0.0963,0.0045
1,Logistic Regression,0.7866,0.0219,0.1026,0.0032


### Interpretación inicial de la validación cruzada

La validación cruzada permite comprobar si el orden observado en la comparación inicial entre modelos se mantiene al variar las particiones de los datos. En esta fase, el interés principal no es solo identificar el mejor promedio, sino también observar la estabilidad del rendimiento entre folds.

Un modelo con media alta pero variabilidad excesiva puede resultar menos fiable que otro ligeramente inferior pero más estable. Por ello, en la discusión se considerarán conjuntamente el rendimiento medio y la dispersión observada.

### Interpretación de la validación cruzada

La validación cruzada confirma que los modelos basados en árboles superan de forma consistente al baseline lineal de regresión logística. No obstante, también muestra que la ventaja observada en una partición única a favor de Random Forest no es tan concluyente cuando el rendimiento se promedia sobre varios folds.

En promedio, Gradient Boosting obtiene el mayor AUC-ROC, mientras que Random Forest presenta el mejor Brier score medio. Dado que ambas diferencias son pequeñas y la variabilidad entre folds es similar, ambos modelos pueden considerarse candidatos competitivos en esta fase.

In [29]:
cv_summary_discussion = cv_summary.copy()
cv_summary_discussion["AUC_intervalo_aprox"] = (
    cv_summary_discussion["AUC_ROC_mean"].round(4).astype(str)
    + " ± "
    + cv_summary_discussion["AUC_ROC_std"].round(4).astype(str)
)
cv_summary_discussion["Brier_intervalo_aprox"] = (
    cv_summary_discussion["Brier_score_mean"].round(4).astype(str)
    + " ± "
    + cv_summary_discussion["Brier_score_std"].round(4).astype(str)
)

display(
    cv_summary_discussion[
        ["modelo", "AUC_intervalo_aprox", "Brier_intervalo_aprox"]
    ]
)

,modelo,AUC_intervalo_aprox,Brier_intervalo_aprox
0,Gradient Boosting,0.8254 ± 0.0205,0.0969 ± 0.005
2,Random Forest,0.8222 ± 0.0222,0.0963 ± 0.0045
1,Logistic Regression,0.7866 ± 0.0219,0.1026 ± 0.0032


### Criterio provisional de selección tras validación cruzada

Tras la validación cruzada, la selección del modelo candidato líder no puede basarse únicamente en una sola métrica. En esta fase se considera que Random Forest y Gradient Boosting son los dos candidatos principales, mientras que la regresión logística queda como baseline interpretable de referencia.

La decisión final entre ambos modelos basados en árboles se pospone hasta incorporar una evaluación adicional orientada al caso de uso clínico, especialmente mediante métricas top-k y análisis de comportamiento sobre el ranking.

## 8) Validación cruzada orientada a ranking clínico

Aunque AUC-ROC y Brier score son útiles para evaluar discriminación y calidad probabilística, el caso de uso de MIMIC-TRIAGE exige además analizar cómo se comportan los modelos al priorizar pacientes.

Por ello, a continuación se amplía la validación cruzada incorporando métricas top-k de ranking clínico. En esta fase se comparan únicamente los dos modelos candidatos principales, Random Forest y Gradient Boosting, mediante Recall@25, Recall@50 y Recall@100.

In [30]:
def cross_validate_models_topk(models, X, y, ks=(25, 50, 100), n_splits=5, random_state=RANDOM_SEED):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    rows = []

    for model_name, model in models.items():
        fold_idx = 1

        for train_idx, val_idx in skf.split(X, y):
            X_train_fold = X.iloc[train_idx]
            X_val_fold = X.iloc[val_idx]
            y_train_fold = y.iloc[train_idx]
            y_val_fold = y.iloc[val_idx]

            fitted_model = clone(model)
            fitted_model.fit(X_train_fold, y_train_fold)
            val_proba = fitted_model.predict_proba(X_val_fold)[:, 1]

            rank_df = ranking_metrics(
                y_true=y_val_fold.to_numpy(),
                y_score=val_proba,
                ks=ks
            ).copy()

            rank_df["modelo"] = model_name
            rank_df["fold"] = fold_idx
            rank_df["prevalencia_fold"] = float(y_val_fold.mean())

            rows.append(rank_df)

            fold_idx += 1

    return pd.concat(rows, ignore_index=True)

In [31]:
topk_models = {
    "Random Forest": cv_models["Random Forest"],
    "Gradient Boosting": cv_models["Gradient Boosting"],
}

list(topk_models.keys())

['Random Forest', 'Gradient Boosting']

In [32]:
cv_topk_results = cross_validate_models_topk(
    models=topk_models,
    X=X_cv,
    y=y_cv,
    ks=(25, 50, 100),
    n_splits=5,
    random_state=RANDOM_SEED
)

display(cv_topk_results.round(4))

,k,deaths_top_k,Recall_at_k,pct_fallecidos_en_top_k,modelo,fold,prevalencia_fold
0,25,14,0.1273,0.56,Random Forest,1,0.1375
1,50,23,0.2091,0.46,Random Forest,1,0.1375
2,100,37,0.3364,0.37,Random Forest,1,0.1375
3,25,17,0.1532,0.68,Random Forest,2,0.1388
4,50,28,0.2523,0.56,Random Forest,2,0.1388
5,100,47,0.4234,0.47,Random Forest,2,0.1388
6,25,17,0.1532,0.68,Random Forest,3,0.1388
7,50,25,0.2252,0.50,Random Forest,3,0.1388
8,100,46,0.4144,0.46,Random Forest,3,0.1388
9,25,13,0.1171,0.52,Random Forest,4,0.1388


In [33]:
cv_topk_summary = (
    cv_topk_results
    .groupby(["modelo", "k"])
    .agg(
        Recall_at_k_mean=("Recall_at_k", "mean"),
        Recall_at_k_std=("Recall_at_k", "std"),
        pct_fallecidos_topk_mean=("pct_fallecidos_en_top_k", "mean"),
        pct_fallecidos_topk_std=("pct_fallecidos_en_top_k", "std"),
    )
    .reset_index()
    .sort_values(["k", "Recall_at_k_mean"], ascending=[True, False])
)

display(cv_topk_summary.round(4))

,modelo,k,Recall_at_k_mean,Recall_at_k_std,pct_fallecidos_topk_mean,pct_fallecidos_topk_std
0,Gradient Boosting,25,0.1462,0.0231,0.648,0.1035
3,Random Forest,25,0.1408,0.0173,0.624,0.0780
4,Random Forest,50,0.2472,0.0411,0.548,0.0923
1,Gradient Boosting,50,0.2346,0.0398,0.520,0.0894
5,Random Forest,100,0.4222,0.0628,0.468,0.0709
2,Gradient Boosting,100,0.4204,0.0519,0.466,0.0590


In [34]:
topk_discussion_table = cv_topk_summary.copy()

topk_discussion_table["Recall@k"] = (
    topk_discussion_table["Recall_at_k_mean"].round(4).astype(str)
    + " ± "
    + topk_discussion_table["Recall_at_k_std"].round(4).astype(str)
)

topk_discussion_table["% fallecidos top-k"] = (
    topk_discussion_table["pct_fallecidos_topk_mean"].round(4).astype(str)
    + " ± "
    + topk_discussion_table["pct_fallecidos_topk_std"].round(4).astype(str)
)

display(
    topk_discussion_table[
        ["modelo", "k", "Recall@k", "% fallecidos top-k"]
    ]
)

,modelo,k,Recall@k,% fallecidos top-k
0,Gradient Boosting,25,0.1462 ± 0.0231,0.648 ± 0.1035
3,Random Forest,25,0.1408 ± 0.0173,0.624 ± 0.078
4,Random Forest,50,0.2472 ± 0.0411,0.548 ± 0.0923
1,Gradient Boosting,50,0.2346 ± 0.0398,0.52 ± 0.0894
5,Random Forest,100,0.4222 ± 0.0628,0.468 ± 0.0709
2,Gradient Boosting,100,0.4204 ± 0.0519,0.466 ± 0.059


### Interpretación de la validación cruzada top-k

Esta validación permite comparar los modelos desde la perspectiva más cercana al caso de uso clínico: cuántos pacientes fallecidos consiguen situar de forma consistente en la parte alta del ranking.

Si uno de los dos modelos muestra ventajas claras y sostenidas en Recall@25, Recall@50 o Recall@100, esa evidencia tendrá un peso importante en la selección final, ya que refleja directamente su utilidad para la priorización de pacientes.

### Interpretación conjunta de las métricas top-k

La comparación top-k muestra que ambos modelos basados en árboles presentan un comportamiento muy cercano, sin un dominador absoluto en todos los cortes del ranking. Gradient Boosting obtiene una ligera ventaja en el top-25, mientras que Random Forest muestra mejores resultados medios en top-50 y top-100.

En conjunto, estas evidencias sugieren que ambos modelos son adecuados para el problema de priorización clínica, aunque Random Forest ofrece un comportamiento algo más equilibrado cuando se consideran conjuntamente discriminación global, calidad probabilística y rendimiento top-k en distintos niveles del ranking.

In [35]:
final_model_decision_table = pd.DataFrame([
    {
        "modelo": "Logistic Regression",
        "rol": "Baseline interpretable",
        "evidencia": "Peor rendimiento global; util como referencia lineal",
    },
    {
        "modelo": "Random Forest",
        "rol": "Candidato principal",
        "evidencia": "Mejor equilibrio entre AUC, Brier y top-k en validacion cruzada",
    },
    {
        "modelo": "Gradient Boosting",
        "rol": "Competidor cercano",
        "evidencia": "Muy competitivo; ligera ventaja en top-25",
    },
])

display(final_model_decision_table)

,modelo,rol,evidencia
0,Logistic Regression,Baseline interpretable,Peor rendimiento global; util como referencia ...
1,Random Forest,Candidato principal,"Mejor equilibrio entre AUC, Brier y top-k en v..."
2,Gradient Boosting,Competidor cercano,Muy competitivo; ligera ventaja en top-25


### Selección provisional del modelo

A la vista del conjunto de resultados obtenidos, Random Forest se selecciona provisionalmente como modelo principal del Hito 3. Esta elección no implica que Gradient Boosting sea descartado, sino que, en esta fase, Random Forest ofrece el mejor equilibrio global entre rendimiento predictivo, utilidad operativa para ranking clínico y estabilidad.

La siguiente etapa consistirá en profundizar sobre este modelo mediante ajuste de hiperparámetros y análisis adicional de interpretabilidad.

## 9) Ajuste de hiperparámetros del modelo seleccionado

Una vez identificado Random Forest como modelo candidato principal, se realiza un ajuste acotado de hiperparámetros para comprobar si su rendimiento puede mejorarse sin alterar la lógica general del experimento.

La búsqueda se plantea de forma intencionadamente reducida, priorizando interpretabilidad metodológica y coste computacional razonable sobre una exploración exhaustiva del espacio de parámetros.

In [36]:
rf_tuning_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        random_state=RANDOM_SEED,
        n_jobs=-1
    ))
])

rf_param_grid = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_leaf": [1, 2, 5],
    "model__max_features": ["sqrt", 0.5],
}

print("Numero total de combinaciones:", 
      len(rf_param_grid["model__n_estimators"]) *
      len(rf_param_grid["model__max_depth"]) *
      len(rf_param_grid["model__min_samples_leaf"]) *
      len(rf_param_grid["model__max_features"]))

Numero total de combinaciones: 54


In [37]:
rf_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

In [38]:
rf_grid_auc = GridSearchCV(
    estimator=rf_tuning_pipeline,
    param_grid=rf_param_grid,
    scoring="roc_auc",
    cv=rf_cv,
    n_jobs=-1,
    refit=True,
    verbose=1
)

rf_grid_auc.fit(X_cv, y_cv)

Fitting 5 folds for each of 54 candidates, totalling 270 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [None, 5, ...], 'model__max_features': ['sqrt', 0.5], 'model__min_samples_leaf': [1, 2, ...], 'model__n_estimators': [200, 300, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more

In [39]:
print("Mejor score CV (AUC):", rf_grid_auc.best_score_)
print("Mejores hiperparametros:")
print(rf_grid_auc.best_params_)

Mejor score CV (AUC): 0.8253183063453777
Mejores hiperparametros:
{'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 5, 'model__n_estimators': 500}


In [40]:
rf_tuning_results = pd.DataFrame(rf_grid_auc.cv_results_)

rf_tuning_summary = (
    rf_tuning_results[
        [
            "mean_test_score",
            "std_test_score",
            "param_model__n_estimators",
            "param_model__max_depth",
            "param_model__min_samples_leaf",
            "param_model__max_features",
            "rank_test_score",
        ]
    ]
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)

display(rf_tuning_summary.head(10))

,mean_test_score,std_test_score,param_model__n_estimators,param_model__max_depth,param_model__min_samples_leaf,param_model__max_features,rank_test_score
0,0.825318,0.020586,500,None,5,sqrt,1
1,0.824972,0.018557,200,None,5,sqrt,2
2,0.824816,0.020011,300,None,5,sqrt,3
3,0.824066,0.021834,200,10,5,sqrt,4
4,0.823470,0.022295,500,10,5,sqrt,5
5,0.822869,0.022352,300,10,5,sqrt,6
6,0.822818,0.022448,500,10,2,sqrt,7
7,0.822283,0.022826,300,10,2,sqrt,8
8,0.822231,0.019881,300,None,2,sqrt,9
9,0.821406,0.020767,500,None,2,sqrt,10


### Interpretación inicial del tuning

El ajuste de hiperparámetros permite comprobar si la configuración inicial del Random Forest era cercana a una solución razonable o si existía margen claro de mejora. En esta fase, el interés principal no es solo encontrar la combinación ganadora, sino también observar si las mejores configuraciones son consistentes entre sí o si el rendimiento depende de ajustes muy específicos.

### Interpretación de los resultados del tuning

El ajuste de hiperparámetros muestra una mejora moderada respecto a la configuración inicial del Random Forest, pero no una ganancia drástica. Esto sugiere que el modelo ya partía de una configuración razonable y que el rendimiento no depende de un ajuste excesivamente fino.

Además, las mejores combinaciones presentan un patrón consistente, especialmente en el uso de `max_features='sqrt'` y valores relativamente conservadores de `min_samples_leaf`. Esta coherencia refuerza la idea de que el comportamiento del modelo es estable y metodológicamente defendible.

In [41]:
best_rf_model = rf_grid_auc.best_estimator_

best_rf_model.fit(X_train_rf, y_train_rf)
test_proba_rf_tuned = best_rf_model.predict_proba(X_test_rf)[:, 1]

ranking_test_rf_tuned = test_df_rf[["RecordID", "In-hospital_death"]].copy()
ranking_test_rf_tuned["pred_proba_mortality"] = test_proba_rf_tuned
ranking_test_rf_tuned = ranking_test_rf_tuned.sort_values("pred_proba_mortality", ascending=False).reset_index(drop=True)

display(ranking_test_rf_tuned.head(20))

,RecordID,In-hospital_death,pred_proba_mortality
0,132960,1,0.720028
1,137325,1,0.687908
2,142160,1,0.683838
3,134580,1,0.665319
4,138530,1,0.645032
5,134835,1,0.629898
6,135305,1,0.620878
7,133780,1,0.617460
8,133975,0,0.613783
9,135210,1,0.607478


In [42]:
rank_metrics_rf_tuned = ranking_metrics(
    y_true=y_test_rf.to_numpy(),
    y_score=test_proba_rf_tuned,
    ks=(25, 50, 100, 200)
)
display(rank_metrics_rf_tuned)

auc_rf_tuned = roc_auc_score(y_test_rf, test_proba_rf_tuned)
brier_rf_tuned = brier_score_loss(y_test_rf, test_proba_rf_tuned)

prob_metrics_rf_tuned = pd.DataFrame(
    {
        "metrica": ["AUC_ROC", "Brier_score", "prevalencia_test"],
        "valor": [auc_rf_tuned, brier_rf_tuned, float(np.mean(y_test_rf))],
    }
)
display(prob_metrics_rf_tuned)

,k,deaths_top_k,Recall_at_k,pct_fallecidos_en_top_k
0,25,16,0.168421,0.640
1,50,31,0.326316,0.620
2,100,44,0.463158,0.440
3,200,63,0.663158,0.315


,metrica,valor
0,AUC_ROC,0.855419
1,Brier_score,0.081489
2,prevalencia_test,0.118012


In [43]:
rf_before_after = pd.DataFrame([
    {
        "modelo": "Random Forest inicial",
        "AUC_ROC": auc_rf,
        "Brier_score": brier_rf,
        "Recall@25": rank_metrics_rf.loc[rank_metrics_rf["k"] == 25, "Recall_at_k"].iloc[0],
        "Recall@50": rank_metrics_rf.loc[rank_metrics_rf["k"] == 50, "Recall_at_k"].iloc[0],
        "Recall@100": rank_metrics_rf.loc[rank_metrics_rf["k"] == 100, "Recall_at_k"].iloc[0],
        "% fallecidos top25": rank_metrics_rf.loc[rank_metrics_rf["k"] == 25, "pct_fallecidos_en_top_k"].iloc[0],
    },
    {
        "modelo": "Random Forest ajustado",
        "AUC_ROC": auc_rf_tuned,
        "Brier_score": brier_rf_tuned,
        "Recall@25": rank_metrics_rf_tuned.loc[rank_metrics_rf_tuned["k"] == 25, "Recall_at_k"].iloc[0],
        "Recall@50": rank_metrics_rf_tuned.loc[rank_metrics_rf_tuned["k"] == 50, "Recall_at_k"].iloc[0],
        "Recall@100": rank_metrics_rf_tuned.loc[rank_metrics_rf_tuned["k"] == 100, "Recall_at_k"].iloc[0],
        "% fallecidos top25": rank_metrics_rf_tuned.loc[rank_metrics_rf_tuned["k"] == 25, "pct_fallecidos_en_top_k"].iloc[0],
    },
])

display(rf_before_after.round(4))

,modelo,AUC_ROC,Brier_score,Recall@25,Recall@50,Recall@100,% fallecidos top25
0,Random Forest inicial,0.8520,0.0818,0.1789,0.2947,0.4632,0.68
1,Random Forest ajustado,0.8554,0.0815,0.1684,0.3263,0.4632,0.64


### Interpretación del impacto del tuning en test

El ajuste de hiperparámetros produce una mejora ligera en las métricas probabilísticas del Random Forest, aumentando el AUC-ROC y reduciendo el Brier score respecto a la configuración inicial. Esto sugiere una mejora real, aunque moderada, en la capacidad global de discriminación y estimación del riesgo.

Sin embargo, el efecto sobre las métricas top-k no es uniforme. Mientras que el modelo ajustado mejora en Recall@50, no supera a la configuración inicial en Recall@25 ni en el porcentaje de fallecidos concentrados en el top-25. Por tanto, el tuning refuerza el rendimiento global del modelo, pero no implica una mejora absoluta en todos los niveles del ranking clínico.

In [44]:
final_model_summary = pd.DataFrame([
    {
        "modelo": "Logistic Regression",
        "estado": "Baseline",
        "AUC_ROC_test": auc_sk,
        "Brier_test": brier_sk,
        "comentario": "Referencia interpretable y lineal",
    },
    {
        "modelo": "Gradient Boosting",
        "estado": "Competidor cercano",
        "AUC_ROC_test": auc_gb,
        "Brier_test": brier_gb,
        "comentario": "Muy competitivo; ligera ventaja en top-25 en CV",
    },
    {
        "modelo": "Random Forest inicial",
        "estado": "Candidato fuerte",
        "AUC_ROC_test": auc_rf,
        "Brier_test": brier_rf,
        "comentario": "Muy buen equilibrio global",
    },
    {
        "modelo": "Random Forest ajustado",
        "estado": "Modelo seleccionado",
        "AUC_ROC_test": auc_rf_tuned,
        "Brier_test": brier_rf_tuned,
        "comentario": "Mejor rendimiento global tras tuning",
    },
])

display(final_model_summary.round(4))

,modelo,estado,AUC_ROC_test,Brier_test,comentario
0,Logistic Regression,Baseline,0.7806,0.0887,Referencia interpretable y lineal
1,Gradient Boosting,Competidor cercano,0.8401,0.0839,Muy competitivo; ligera ventaja en top-25 en CV
2,Random Forest inicial,Candidato fuerte,0.8520,0.0818,Muy buen equilibrio global
3,Random Forest ajustado,Modelo seleccionado,0.8554,0.0815,Mejor rendimiento global tras tuning


## Conclusión del modelado

A partir del conjunto de experimentos realizados, el Random Forest ajustado se selecciona como modelo principal del Hito 3. La regresión logística queda como baseline interpretable de referencia, mientras que Gradient Boosting se mantiene como alternativa muy competitiva.

La elección del modelo final se basa en el mejor equilibrio observado entre discriminación global, calidad probabilística y comportamiento en ranking clínico, sin perder de vista que algunas métricas top-k muy tempranas presentan diferencias pequeñas entre configuraciones.

## 10) Interpretabilidad del modelo final

Una vez seleccionado provisionalmente el Random Forest ajustado como modelo principal, se analiza qué variables contribuyen en mayor medida a sus decisiones. En esta primera fase, la interpretabilidad se aborda desde una perspectiva global mediante la importancia de variables del modelo.

Este análisis no explica por sí solo cada predicción individual, pero sí permite identificar qué factores clínicos tienen mayor peso promedio en el comportamiento del sistema.

In [45]:
# Importancia global de variables del Random Forest ajustado

feature_importances = best_rf_model.named_steps["model"].feature_importances_

rf_importance_df = pd.DataFrame({
    "feature": features,
    "importance": feature_importances
}).sort_values("importance", ascending=False).reset_index(drop=True)

display(rf_importance_df)

,feature,importance
0,GCS_mean,0.168243
1,Urine_mean,0.119485
2,BUN_mean,0.105815
3,WBC_mean,0.084329
4,Temp_mean,0.082397
5,HR_mean,0.074775
6,Creatinine_mean,0.074765
7,Age_first,0.073859
8,SysABP_mean,0.072635
9,MAP_mean,0.047980


In [46]:
rf_top15_importance = rf_importance_df.head(15).copy()
display(rf_top15_importance)

,feature,importance
0,GCS_mean,0.168243
1,Urine_mean,0.119485
2,BUN_mean,0.105815
3,WBC_mean,0.084329
4,Temp_mean,0.082397
5,HR_mean,0.074775
6,Creatinine_mean,0.074765
7,Age_first,0.073859
8,SysABP_mean,0.072635
9,MAP_mean,0.047980


In [47]:
fig, ax = plt.subplots(figsize=(10, 6))
top_plot = rf_importance_df.head(15).iloc[::-1]

ax.barh(top_plot["feature"], top_plot["importance"])
ax.set_title("Variables más importantes - Random Forest ajustado")
ax.set_xlabel("Importancia")
ax.set_ylabel("Variable")

fig.tight_layout()
importance_path = FIGURES_DIR / "rf_tuned_feature_importance_top.png"
fig.savefig(importance_path, dpi=160)
plt.close(fig)

print(f"Figura guardada en: {importance_path}")
display(rf_top15_importance)

Figura guardada en: C:\Users\Natalia\Proyecto-DISIA\Hito 3\figures\rf_tuned_feature_importance_top.png


,feature,importance
0,GCS_mean,0.168243
1,Urine_mean,0.119485
2,BUN_mean,0.105815
3,WBC_mean,0.084329
4,Temp_mean,0.082397
5,HR_mean,0.074775
6,Creatinine_mean,0.074765
7,Age_first,0.073859
8,SysABP_mean,0.072635
9,MAP_mean,0.047980


### Interpretación inicial de la importancia de variables

La importancia global de variables permite identificar qué señales clínicas tienen mayor peso promedio en el comportamiento del modelo. Este análisis resulta útil para comprobar si las variables destacadas por el Random Forest son coherentes con la intuición clínica y con el conocimiento del dominio.

No obstante, esta medida debe interpretarse con cautela, ya que resume el comportamiento global del modelo y no explica por sí sola la decisión tomada en cada paciente concreto.

### Lectura clínica preliminar de las variables más relevantes

Las variables más importantes del modelo ajustado apuntan a dimensiones clínicas plausibles del deterioro en UCI. En primer lugar, `GCS_mean` destaca como la variable de mayor peso, lo que sugiere que el estado neurológico medio del paciente durante las primeras 48 horas tiene una influencia destacada en la estimación del riesgo.

Junto a ella, variables relacionadas con función renal y balance hídrico como `Urine_mean`, `BUN_mean` y `Creatinine_mean` aparecen entre las más relevantes, lo que resulta coherente con la importancia pronóstica de la perfusión y del daño orgánico en pacientes críticos. También destacan `WBC_mean` y `Temp_mean`, compatibles con procesos infecciosos o inflamatorios, así como variables hemodinámicas como `HR_mean`, `SysABP_mean` y `MAP_mean`.

En conjunto, el modelo parece apoyarse en señales fisiológicas y clínicas razonables, lo que refuerza su plausibilidad desde el punto de vista del dominio.

## Interpretabilidad avanzada con SHAP

Como complemento a la importancia global de variables, se incorpora un análisis basado en SHAP para obtener una explicación más fina del comportamiento del modelo. A diferencia de la importancia agregada del Random Forest, SHAP permite estimar la contribución de cada variable a la predicción realizada para cada observación.

En esta primera fase, el objetivo es analizar el comportamiento global del modelo sobre una muestra representativa del conjunto de test, sin entrar todavía en casos individuales concretos.

In [48]:
# Muestra pequeña para SHAP sobre el conjunto de test

X_test_rf_imputed = pd.DataFrame(
    best_rf_model.named_steps["imputer"].transform(X_test_rf),
    columns=features,
    index=X_test_rf.index
)

shap_sample_size = min(200, len(X_test_rf_imputed))
X_shap_sample = X_test_rf_imputed.sample(n=shap_sample_size, random_state=RANDOM_SEED)

print("Shape muestra SHAP:", X_shap_sample.shape)
display(X_shap_sample.head())

Shape muestra SHAP: (200, 14)


,Age_first,Gender_first,ICUType_first,GCS_mean,HR_mean,MAP_mean,SysABP_mean,DiasABP_mean,RespRate_mean,Temp_mean,Creatinine_mean,BUN_mean,WBC_mean,Urine_mean
980,84.0,0.0,1.0,15.000000,67.183333,73.740000,100.686275,43.431373,19.117647,36.836364,0.600,16.166667,4.240000,199.178571
3568,72.0,1.0,4.0,9.000000,92.666667,75.000000,117.446809,56.808511,19.117647,37.592308,1.575,40.500000,12.833333,49.812500
863,74.0,0.0,4.0,8.666667,105.096154,78.820513,108.789474,54.500000,19.117647,37.851923,1.025,14.750000,16.220000,147.239130
2627,39.0,0.0,3.0,15.000000,92.343750,78.830957,116.986667,58.807018,20.451613,37.166667,0.800,13.666667,16.833333,615.000000
2705,71.0,1.0,1.0,15.000000,85.025000,90.220779,115.076923,64.358974,19.117647,36.954545,0.900,20.750000,11.266667,187.391304


In [49]:
rf_model_only = best_rf_model.named_steps["model"]

explainer = shap.TreeExplainer(rf_model_only)
shap_values = explainer.shap_values(X_shap_sample)

print(type(shap_values))

<class 'numpy.ndarray'>


In [50]:
shap_array = np.array(shap_values)

if isinstance(shap_values, list):
    shap_values_positive = np.array(shap_values[1])
elif shap_array.ndim == 3 and shap_array.shape[2] == 2:
    shap_values_positive = shap_array[:, :, 1]
else:
    shap_values_positive = shap_array

print("Shape SHAP clase positiva:", shap_values_positive.shape)

Shape SHAP clase positiva: (200, 14)


In [51]:
shap_importance_df = pd.DataFrame({
    "feature": features,
    "mean_abs_shap": np.abs(shap_values_positive).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

display(shap_importance_df)

,feature,mean_abs_shap
0,GCS_mean,0.050810
1,BUN_mean,0.032714
2,Urine_mean,0.030937
3,Age_first,0.020221
4,Temp_mean,0.017972
5,ICUType_first,0.015284
6,SysABP_mean,0.014236
7,HR_mean,0.014192
8,WBC_mean,0.012968
9,Creatinine_mean,0.011929


In [52]:
fig, ax = plt.subplots(figsize=(10, 6))
top_shap_plot = shap_importance_df.head(15).iloc[::-1]

ax.barh(top_shap_plot["feature"], top_shap_plot["mean_abs_shap"])
ax.set_title("Importancia global media según SHAP - Random Forest ajustado")
ax.set_xlabel("mean(|SHAP value|)")
ax.set_ylabel("Variable")

fig.tight_layout()
shap_bar_path = FIGURES_DIR / "rf_tuned_shap_mean_abs_top.png"
fig.savefig(shap_bar_path, dpi=160)
plt.close(fig)

print(f"Figura SHAP guardada en: {shap_bar_path}")
display(shap_importance_df.head(15))

Figura SHAP guardada en: C:\Users\Natalia\Proyecto-DISIA\Hito 3\figures\rf_tuned_shap_mean_abs_top.png


,feature,mean_abs_shap
0,GCS_mean,0.050810
1,BUN_mean,0.032714
2,Urine_mean,0.030937
3,Age_first,0.020221
4,Temp_mean,0.017972
5,ICUType_first,0.015284
6,SysABP_mean,0.014236
7,HR_mean,0.014192
8,WBC_mean,0.012968
9,Creatinine_mean,0.011929


### Interpretación inicial del análisis SHAP

El resumen basado en valores SHAP permite estimar la contribución media absoluta de cada variable al score predicho por el modelo. A diferencia de la importancia interna del Random Forest, esta medida está más directamente conectada con el impacto real de cada variable sobre las predicciones generadas.

En esta fase, el interés principal es comprobar si las variables destacadas por SHAP son consistentes con las obtenidas mediante importancia global del modelo y si refuerzan la plausibilidad clínica del sistema.

### Lectura conjunta de SHAP e importancia global

El análisis SHAP confirma en gran medida el patrón observado mediante la importancia interna del Random Forest. `GCS_mean` vuelve a aparecer como la variable más influyente, y también destacan variables relacionadas con función renal y balance hídrico (`BUN_mean`, `Urine_mean`, `Creatinine_mean`), junto con edad, temperatura y variables hemodinámicas.

La consistencia entre ambos enfoques refuerza la idea de que el modelo se apoya en señales clínicamente plausibles y no en patrones arbitrarios. Además, la baja contribución de `Gender_first` sugiere que esta variable apenas condiciona las predicciones del sistema en comparación con otros marcadores fisiológicos.

In [53]:
importance_compare = rf_importance_df.merge(
    shap_importance_df,
    on="feature",
    how="inner"
).sort_values("importance", ascending=False).reset_index(drop=True)

display(importance_compare)

,feature,importance,mean_abs_shap
0,GCS_mean,0.168243,0.050810
1,Urine_mean,0.119485,0.030937
2,BUN_mean,0.105815,0.032714
3,WBC_mean,0.084329,0.012968
4,Temp_mean,0.082397,0.017972
5,HR_mean,0.074775,0.014192
6,Creatinine_mean,0.074765,0.011929
7,Age_first,0.073859,0.020221
8,SysABP_mean,0.072635,0.014236
9,MAP_mean,0.047980,0.005989


In [54]:
display(importance_compare.head(10))

,feature,importance,mean_abs_shap
0,GCS_mean,0.168243,0.050810
1,Urine_mean,0.119485,0.030937
2,BUN_mean,0.105815,0.032714
3,WBC_mean,0.084329,0.012968
4,Temp_mean,0.082397,0.017972
5,HR_mean,0.074775,0.014192
6,Creatinine_mean,0.074765,0.011929
7,Age_first,0.073859,0.020221
8,SysABP_mean,0.072635,0.014236
9,MAP_mean,0.047980,0.005989


## 11) Conclusiones

En este Hito 3 se ha partido del dataset procesado por estancia construido a partir de las primeras 48 horas de monitorización en Set A, y se ha validado experimentalmente su utilidad para una tarea de scoring probabilístico y ranking clínico de mortalidad intrahospitalaria.

La regresión logística se utilizó como baseline interpretable, mientras que Random Forest y Gradient Boosting se evaluaron como modelos no lineales alternativos. Los resultados mostraron una mejora clara de los modelos basados en árboles respecto al baseline lineal. Tras la comparación inicial, la validación cruzada y el ajuste acotado de hiperparámetros, el Random Forest ajustado se seleccionó como modelo principal por ofrecer el mejor equilibrio global entre discriminación, calidad probabilística y utilidad operativa en ranking clínico.

Finalmente, el análisis de interpretabilidad mediante importancia global de variables y SHAP mostró que el modelo se apoya en señales clínicamente plausibles, especialmente relacionadas con estado neurológico, función renal, respuesta inflamatoria y estabilidad hemodinámica.